# Lab 2 — Prompt Agent

A **prompt agent** is a declarative agent (model + instructions) that Foundry hosts for you — no container, no Docker. In this lab you will:

1. Create a prompt agent with the `azure-ai-projects` SDK.
2. Invoke it through the OpenAI-compatible Responses API.
3. Add a new **version** with updated instructions.

> Make sure you completed **Lab 1** first.

In [1]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

load_dotenv()

project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1")

project = AIProjectClient(endpoint=project_endpoint, credential=DefaultAzureCredential())
AGENT_NAME = "labs-prompt-agent"

## 1. Create the prompt agent

`create_version` registers the agent definition (model + instructions). The first call creates version `1`.

In [2]:
agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a concise, helpful assistant. "
            "Answer in no more than three sentences."
        ),
    ),
)
print(f"Created agent '{agent.name}' version {agent.version} (model: {model})")

Created agent 'labs-prompt-agent' version 3 (model: gpt-5.4-mini)


## 2. Invoke the agent

We invoke the agent by reference through the OpenAI-compatible Responses API.

In [3]:
openai_client = project.get_openai_client()

response = openai_client.responses.create(
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="In one sentence, what is Azure AI Foundry?",
)
print(response.output_text)

Azure AI Foundry is Microsoft’s platform for building, customizing, evaluating, and deploying AI applications and agents using Azure’s models, tools, and cloud infrastructure.


## 3. Add a new version

Re-running `create_version` with different instructions produces a new version. Foundry serves the latest active version automatically.

In [4]:
agent_v2 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a pirate-themed assistant. Answer briefly and in pirate speak."
        ),
    ),
)
print(f"New version: {agent_v2.version}")

response = openai_client.responses.create(
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="Say hello.",
)
print(response.output_text)

New version: 4
Ahoy! Hello there, matey!


## 4. Ground the prompt agent with RAG (Azure AI Search)

A prompt agent can do **Retrieval Augmented Generation** . You attach an `AzureAISearchTool` to the agent definition and Foundry queries your index for you before each model call, returning answers with inline citations.

The tool references the search index through a **project connection** (resolved by name).

First we **provision and seed a small search index** so there's something to ground against. Each document carries a unique *canary* token we can look for in the answer to prove grounding.

> **Prerequisites:** an **Azure AI Search project connection** in your Foundry project. Set `AZURE_SEARCH_ENDPOINT`, `AZURE_SEARCH_CONNECTION_NAME`, and `AZURE_SEARCH_INDEX_NAME` in `.env`. Your signed-in identity needs **Search Service Contributor** + **Search Index Data Contributor** on the search service (RBAC can take a few minutes to propagate).


In [5]:
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchableField, SearchFieldDataType, SearchIndex, SimpleField,
)

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_index_name = os.environ.get("AZURE_SEARCH_INDEX_NAME", "contoso-outdoors")
credential = DefaultAzureCredential()

DOCUMENTS = [
    {"id": "return-policy", "sourceName": "Contoso Outdoors Return Policy",
     "sourceLink": "https://contoso.com/policies/returns",
     "content": "Customers may return any item within 30 days of delivery. Items should be "
                "unused and include original packaging. Every accepted return ships back with "
                "a complimentary Contoso TrailRunner sticker pack, item code TR-CANARY-7821."},
    {"id": "shipping-guide", "sourceName": "Contoso Outdoors Shipping Guide",
     "sourceLink": "https://contoso.com/help/shipping",
     "content": "Standard shipping is free on orders over $50 and arrives in 3-5 business days. "
                "Use promo code SHIP-CANARY-4493 for a one-time free overnight upgrade."},
    {"id": "tent-care", "sourceName": "TrailRunner Tent Care Instructions",
     "sourceLink": "https://contoso.com/manuals/trailrunner-tent",
     "content": "Clean tent fabric with lukewarm water and non-detergent soap. Replacement "
                "waterproofing kits are stocked under SKU TENT-CANARY-9067."},
]

index = SearchIndex(
    name=search_index_name,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String, analyzer_name="standard.lucene"),
        SimpleField(name="sourceName", type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="sourceLink", type=SearchFieldDataType.String),
    ],
)

index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
try:
    index_client.get_index(search_index_name)
    print(f"Index '{search_index_name}' already exists; leaving schema as-is.")
except ResourceNotFoundError:
    index_client.create_index(index)
    print(f"Created index '{search_index_name}'.")

search_client = SearchClient(endpoint=search_endpoint, index_name=search_index_name, credential=credential)
results = search_client.merge_or_upload_documents(documents=DOCUMENTS)
print(f"Uploaded {sum(1 for r in results if r.succeeded)} / {len(DOCUMENTS)} documents.")


Created index 'contoso-outdoors'.
Uploaded 3 / 3 documents.


In [ ]:
from azure.ai.projects.models import (
    AzureAISearchTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

search_connection_name = os.environ["AZURE_SEARCH_CONNECTION_NAME"]
search_index_name = os.environ.get("AZURE_SEARCH_INDEX_NAME", "contoso-outdoors")

# Resolve the project connection id from its friendly name
search_connection = project.connections.get(search_connection_name)

agent_rag = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a helpful support specialist for Contoso Outdoors. "
            "You can ONLY answer questions using information retrieved from the Azure AI Search tool. "
            "ALWAYS call the Azure AI Search tool first, before answering any question — never answer "
            "from your own general knowledge, and never say you lack information without searching first. "
            "If the search returns relevant content, answer using it and always cite your sources, "
            "rendering citations as `[message_idx:search_idx†source]`. "
            "If the search returns nothing relevant, or the question is outside the Contoso Outdoors "
            "knowledge base (e.g. general trivia, creative writing, or unrelated topics), politely refuse: "
            "say you don't have that information in the knowledge base and do not attempt to answer."
        ),
        tools=[
            AzureAISearchTool(
                azure_ai_search=AzureAISearchToolResource(
                    indexes=[
                        AISearchIndexResource(
                            project_connection_id=search_connection.id,
                            index_name=search_index_name,
                            query_type=AzureAISearchQueryType.SIMPLE,
                        ),
                    ]
                )
            )
        ],
    ),
)
print(f"Created RAG agent version {agent_rag.version} (index: {search_index_name})")


Created RAG agent version 5 (index: contoso-outdoors)


Invoke the grounded agent and stream the answer. We set `tool_choice="required"` so the agent must consult the index, then print any `url_citation` annotations. A grounded reply about the return policy includes the **canary token** `TR-CANARY-7821` from the seeded document.


In [16]:
stream_response = openai_client.responses.create(
    stream=True,
    tool_choice="required",
    input="What is your return policy? Include any item codes mentioned.",
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
)

for event in stream_response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.output_item.done":
        if event.item.type == "message":
            text_content = event.item.content[-1]
            if text_content.type == "output_text":
                for annotation in text_content.annotations:
                    if annotation.type == "url_citation":
                        print(
                            f"\nURL citation: {annotation.url} "
                            f"[{annotation.start_index}:{annotation.end_index}]"
                        )
    elif event.type == "response.completed":
        print(f"\n\nFull response:\n{event.response.output_text}")


Contoso Outdoors allows customers to return any item within 30 days of delivery, as long as it’s unused and includes the original packaging【5:0†source】. Accepted returns also include a complimentary Contoso TrailRunner sticker pack, item code TR-CANARY-7821【5:0†source】.
URL citation: https://brk445-search.search.windows.net/ [139:151]

URL citation: https://brk445-search.search.windows.net/ [257:269]


Full response:
Contoso Outdoors allows customers to return any item within 30 days of delivery, as long as it’s unused and includes the original packaging【5:0†source】. Accepted returns also include a complimentary Contoso TrailRunner sticker pack, item code TR-CANARY-7821【5:0†source】.


Prompt agents are perfect when your logic is *instructions + model* — and as section 4 showed, they can even do **RAG** through a Foundry-managed `AzureAISearchTool` (Foundry handles the search call and the agent-identity access for you).

When you need **custom code or your own container** — for example to run retrieval, post-processing, or tools you control end-to-end — use a hosted agent. Continue to **Lab 4**, then compare the container-based RAG approach in **Lab 5**.
